In [51]:
import pandas as pd
import numpy as np
import difflib

# require pip installs
from thefuzz import process 
# from faker import Faker
# fake = Faker()

## fuzzy matching

* this is helpful if you're trying to merge data but the values in the match column don't exactly match
* in this example, our main goal is to add one or more columns from "country_codes" to "essential_indicators"

In [52]:
df = pd.read_csv('data/essential_indicators_messy.csv', index_col=0)

df.columns
list1 = df['Country']
print(list1.shape)
list1.head()

(217,)


0       Afghanistan
1           Albania
2           Algeria
3    American Samoa
4           Andorra
Name: Country, dtype: object

In [53]:
df = pd.read_csv('data/country_codes.csv')#, index_col=0)
df.head()

df.columns
list2 = df['name']
print(list2.shape)
list2.head()

(249,)


0       Afghanistan
1     Åland Islands
2           Albania
3           Algeria
4    American Samoa
Name: name, dtype: object

In [54]:
# count the number of exact matches
matches = list1.isin(list2).sum()
matches

np.int64(181)

In [55]:
# Find names in list1 that aren't in list2
missing = list1[~list1.isin(list2)]
print(len(missing))
missing.head()


36


13        Bahamas, The
23             Bolivia
38     Channel Islands
43    Congo, Dem, Rep,
44         Congo, Rep,
Name: Country, dtype: object

## using the Fuzz

In [56]:
# match missing names to cc names using thefuzz
def find_closest_match_fuzz(name, choices):
    match, score = process.extractOne(name, choices)
    return match, score# if score > 80 else None

In [57]:
find_closest_match_fuzz("Bolivia", list2.to_list())

('Bolivia, Plurinational State of', 90)

In [93]:
matches, scores = zip(*[find_closest_match_fuzz(n,list2.to_list()) for n in missing])
# scores = [get_score(n,m) for n,m in zip(missing,matches)]

match_table = pd.DataFrame({
    'Name':missing,
    'Match':matches,
    'Score':scores
})

print(len(match_table))

# see the worst matches
match_table.sort_values('Score').head(10)

36


,Name,Match,Score
102,Kosovo,Solomon Islands,54
179,"St, Lucia",Saint Lucia,80
89,"Iran, Islamic Rep,","Iran, Islamic Republic of",81
38,Channel Islands,Cocos (Keeling) Islands,86
105,Lao PDR,Lao People's Democratic Republic,86
170,Slovak Republic,Central African Republic,86
125,"Micronesia, Fed, Sts,","Micronesia, Federated States of",86
104,Kyrgyz Republic,Central African Republic,86
207,"Venezuela, RB","Venezuela, Bolivarian Republic of",86
210,West Bank and Gaza,"Bonaire, Sint Eustatius and Saba",86
